<a href="https://colab.research.google.com/github/gjardim27/NewCurve/blob/claude%2Facoustic-streaming-piv-m3hdpa/Classes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup


In [ ]:
# @title color palettes
# Wes Anderson Palettes (ported from https://github.com/karthik/wesanderson)

wa_palettes = {
  "BottleRocket1": ("#A42820", "#5F5647", "#9B110E", "#3F5151", "#4E2A1E", "#550307", "#0C1707"),
  "BottleRocket2": ("#FAD510", "#CB2314", "#273046", "#354823", "#1E1E1E"),
  "Rushmore1": ("#E1BD6D", "#EABE94", "#0B775E", "#35274A" ,"#F2300F"),
  "Rushmore": ("#E1BD6D", "#EABE94", "#0B775E", "#35274A" ,"#F2300F"),
  "Royal1": ("#899DA4", "#C93312", "#FAEFD1", "#DC863B"),
  "Royal2": ("#9A8822", "#F5CDB4", "#F8AFA8", "#FDDDA0", "#74A089"),
  "Zissou1": ("#3B9AB2", "#78B7C5", "#EBCC2A", "#E1AF00", "#F21A00"),
  "Zissou1Continuous": ("#3A9AB2", "#6FB2C1", "#91BAB6", "#A5C2A3", "#BDC881", "#DCCB4E", "#E3B710", "#E79805", "#EC7A05", "#EF5703", "#F11B00"),
  "Darjeeling1": ("#FF0000", "#00A08A", "#F2AD00", "#F98400", "#5BBCD6"),
  "Darjeeling2": ("#ECCBAE", "#046C9A", "#D69C4E", "#ABDDDE", "#000000"),
  "Chevalier1": ("#446455", "#FDD262", "#D3DDDC", "#C7B19C"),
  "FantasticFox1": ("#DD8D29", "#E2D200", "#46ACC8", "#E58601", "#B40F20"),
  "Moonrise1": ("#F3DF6C", "#CEAB07", "#D5D5D3", "#24281A"),
  "Moonrise2": ("#798E87", "#C27D38", "#CCC591", "#29211F"),
  "Moonrise3": ("#85D4E3", "#F4B5BD", "#9C964A", "#CDC08C", "#FAD77B"),
  "Cavalcanti1": ("#D8B70A", "#02401B", "#A2A475", "#81A88D", "#972D15"),
  "GrandBudapest1": ("#F1BB7B", "#FD6467", "#5B1A18", "#D67236"),
  "GrandBudapest2": ("#E6A0C4", "#C6CDF7", "#D8A499", "#7294D4"),
  "IsleofDogs1": ("#9986A5", "#79402E", "#CCBA72", "#0F0D0E", "#D9D0D3", "#8D8680"),
  "IsleofDogs2": ("#EAD3BF", "#AA9486", "#B6854D", "#39312F", "#1C1718"),
  "FrenchDispatch": ("#90D4CC", "#BD3027", "#B0AFA2", "#7FC0C6", "#9D9C85"),
  "AsteroidCity1": ("#0A9F9D", "#CEB175", "#E54E21", "#6C8645", "#C18748"),
  "AsteroidCity2": ("#C52E19", "#AC9765", "#54D8B1", "#b67c3b", "#175149", "#AF4E24"),
  "AsteroidCity3": ("#FBA72A", "#D3D4D8", "#CB7A5C", "#5785C1"),
}


In [ ]:
# @title Setup

# Install packages with pip
# %pip install pint
%pip install pyfluids
# %pip install scipy
%pip install lvpyio==1.3.1


# @title Import dependencies
import gspread
import google.colab, google.auth
from google.colab import drive
import matplotlib.pyplot as plt
import matplotlib.patches
import matplotlib.lines
import matplotlib
import numpy as np
import os
import pandas as pd
import lvpyio as lv
import pyfluids
import datetime
import time
import scipy
import yaml
import scipy.optimize
import scipy.interpolate
import scipy.integrate

dpi = 1000  # @param{type:"integer"}

plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600

# Set defaults for font, labels... etc
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
# plt.rc('axes', labelsize=16)
plt.rc('axes', labelsize=16)  # for presentations
plt.rc('lines', markersize=6, linewidth=1, markeredgecolor = 'none')
plt.rc('text.latex', preamble=r'\usepackage{amsmath}')

# plt.rc('xtick', labelsize=12)
# plt.rc('ytick', labelsize=12)


plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'


# Mount google drive (must be separate cell to avoid re-mounting errors)
# allows reading of text/.csv files from google drive, but NOT google sheets files.
force_remount = False # @param {type:"boolean"}
drive.mount('/content/drive', force_remount=force_remount)

USC_red = np.array([153, 27, 30])/255
USC_yellow = np.array([255, 204, 0])/255

# New Section

In [ ]:
# @title Import PIV classes (definitions now live in src/piv_speaker.py)
# Clones (or updates) the GitHub repo into /content and imports the module.
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/gjardim27/NewCurve.git"
BRANCH = "claude/acoustic-streaming-piv-m3hdpa"
REPO_DIR = "/content/NewCurve"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)

src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import piv_speaker
importlib.reload(piv_speaker)  # pick up edits after a git pull
from piv_speaker import (
    RHO,
    PIVSpeakerInterlaced,
    PIVSpeakerPhaseAverage,
    dual_gaus_third_poly_w_params,
    planar_dual_gauss_third_poly_w_params,
    smallestbox,
)

PME = PIVSpeakerPhaseAverage()